In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from sklearn.impute import SimpleImputer

In [ ]:
data_train = pd.read_csv("train.csv")
X = data_train.drop(columns=["label"])
y = data_train["label"]


columns = X.columns
imputer = SimpleImputer(strategy="constant", fill_value="")
X = imputer.fit_transform(X)
X = pd.DataFrame(X, columns=columns)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=11)

In [ ]:
data_test = pd.read_csv("test.csv")
X_test_kaggle = data_test
test_ids = X_test_kaggle["id"]

In [ ]:
X_na = X[X.isna().any(axis=1)]
fake_row = pd.DataFrame({"id": [11111], "title": "", "body": ""})

X_na = pd.concat([X_na, fake_row], ignore_index=True)

imputer = SimpleImputer(strategy="constant", fill_value="")
X_na = imputer.fit_transform(X_na)
X_na = pd.DataFrame(X_na)
X_na = X_na.drop(2074)

X_na.columns = X.columns
X_na["text"] = X_na["title"] + " " + X_na["title"]

X_na.head(5)

,id,title,body,text
0,2,"Don't know if this is allowed, I work in youth...",,"Don't know if this is allowed, I work in youth..."
1,16,5'7 is one of those heights where to some it's...,,5'7 is one of those heights where to some it's...
2,18,music is the only thing helping me cope with t...,,music is the only thing helping me cope with t...
3,29,Song recommendation number 2 for bored ppl,,Song recommendation number 2 for bored ppl Son...
4,30,Pull out your phone; what is your most used em...,,Pull out your phone; what is your most used em...


In [ ]:
pipeline = Pipeline([
    ("tf-idf", TfidfVectorizer(
        ngram_range=(1, 3),
        max_features=10000,
        min_df=1, 
        max_df=0.9,
        lowercase=True,
        stop_words=None
    )),
    ("classdifier", LogisticRegression(max_iter=1000)),
])

In [ ]:
# X_test = X_test.drop(columns=["id"])
X_test["text"] = X_test["title"] + " " + X_test["body"]
X_test = X_test["text"]
len(X_test)

2160

In [ ]:
# X_test = X_test.drop(columns=["id"])
X_train["text"] = X_train["title"] + " " + X_train["body"]
X_train = X_train["text"]
X_train

1824     Heres little story about my GoOd Day So i woke...
4183     Oh look at that, I fucked up again. And again....
5565              SECOND UPDATE TO "I have a hard choice" 
9411     And now, the six merry murderesses of the Cook...
4694     [Social]He asked him what my plans were. Smoot...
                               ...                        
4023     Depression I spent over an hour writing someth...
7259     How do I get the age thing under my name? I se...
5200     What do i do with my life I turn 17 in  4 mont...
3775     Everything is changing, and that terrifies me....
10137    According to redditmetis I've commented 1k tim...
Name: text, Length: 8640, dtype: str

In [96]:
X["text"] = X["title"] + " " + X["body"]
X = X["text"]

In [101]:
X_test_kaggle["text"] = X_test_kaggle["title"] + " " + X_test_kaggle["body"]
X_test_kaggle = X_test_kaggle["text"]

In [92]:
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.93      0.98      0.95      1754
         1.0       0.86      0.66      0.75       406

    accuracy                           0.92      2160
   macro avg       0.89      0.82      0.85      2160
weighted avg       0.91      0.92      0.91      2160



In [93]:
len(y_pred)

2160

In [102]:
pipeline.fit(X, y)

y_pred = pipeline.predict(X_test_kaggle)

In [103]:
len(y_pred)

973

In [104]:
submission = pd.DataFrame({
    "id": test_ids,
    "price_target": y_pred
})

submission.to_csv('submission.csv', index=False)
submission.head()

,id,price_target
0,0,0.0
1,1,0.0
2,2,0.0
3,3,0.0
4,4,1.0


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import numpy as np

# 1. Создаем пайплайн (используем ваше имя 'classdifier')
pipeline = Pipeline([
    ("tf-idf", TfidfVectorizer(
        ngram_range=(1, 3),
        max_features=10000,
        min_df=1, 
        max_df=0.9,
        lowercase=True,
        stop_words=None
    )),
    ("classdifier", LogisticRegression(max_iter=1000)),
])

# 2. Определяем сетку параметров
# Обратите внимание на префиксы 'tf-idf__' и 'classdifier__'
param_grid = {
    # Параметры для TfidfVectorizer
    'tf-idf__max_features': [5000, 10000, 20000, 50000, 100000],
    'tf-idf__ngram_range': [(1, 2), (1, 3), (1, 4), (1, 5), (1, 10)],
    'tf-idf__min_df': [1, 2, 5],
    'tf-idf__max_df': [0.8, 0.9, 0.95],
    'stop_words': [None, 'english'],
    
    # Параметры для LogisticRegression
    'classdifier__C': [0.1, 0.5, 1, 5, 10, 50, 100],  # Сила регуляризации (обратная lambda)
    'classdifier__solver': ['lbfgs', 'liblinear'], # Алгоритм оптимизации
    'classdifier__penalty': ['l2'], # Для lbfgs только l2
}

# 3. Инициализируем GridSearchCV
# cv=5 означает 5-кратную кросс-валидацию
# scoring='f1_weighted' хорош для несбалансированных данных (как у вас в предыдущем вопросе)
grid_search = GridSearchCV(
    pipeline, 
    param_grid, 
    cv=5, 
    scoring='f1_weighted', 
    n_jobs=-1,  # Использовать все ядра процессора
    verbose=1   # Показывать прогресс
)

# 4. Запускаем поиск (X_train, y_train - ваши данные)
grid_search.fit(X_train, y_train)